# 10 — M3OT ile ince ayar (F4 Görev 15B)

Çalışma zamanı → **GPU** (A100 ya da L4). Secrets: `KAGGLE_API_TOKEN`, `GITHUB_TOKEN`.
Tasarım: `docs/design/faz4-kapsama.md` §3 B. **M3OT test dizileri bu defterde hiç kullanılmaz.**

**Tümünü çalıştır değil.** İki bölüm:
- **Bölüm A — hücre 1-5 (≈ 1-1,5 saat):** M3OT indirme, train dönüştürme, öğretmen etiketleri, kör denetim
  sayfası → Drive. Hücre 5'ten sonra **dur**.
- **Bölüm B — hücre 6-10 (eğitim birkaç saat):** Claude'un verdiği kararla etiketler, eğitim, ölçüm.
  Oturum koparsa hücre 1-2'yi, sonra kaldığın hücreyi yeniden çalıştır: dönüştürülmüş veri ve eğitim
  Drive'dan sürer.

Her hücrenin son satırlarını Claude'a getir.

In [ ]:
# ≈ 1 dk · ayarlar
BRANCH = 'f4'
DRIVE = '/content/drive/MyDrive/baykus'
OUT = '/content/m3ot'                 # dönüştürülmüş M3OT (train)
DV = '/content/dv_rgbt'               # DroneVehicle + birleştirilmiş M3OT train
CACHE = f'{DRIVE}/m3ot/m3ot_train.zip'  # Bölüm B ve kopan oturum için dönüştürülmüş train
E3C_RUN = '2026-10-15_e3c-modality-dropout_s_d4e1280'
import os
NPROC = os.cpu_count()
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(f'{DRIVE}/m3ot', exist_ok=True)

In [ ]:
%%time
# ≈ 3 dk · kurulum
import os, sys
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/OgzhnTarhn/baykus.git' if tok else 'https://github.com/OgzhnTarhn/baykus.git'
if not os.path.isdir('/content/baykus/.git'):
    !git clone -q {url} /content/baykus
!git -C /content/baykus fetch -q {url} {BRANCH}
!git -C /content/baykus checkout -q -B {BRANCH} FETCH_HEAD
!git -C /content/baykus remote set-url origin https://github.com/OgzhnTarhn/baykus.git
%cd /content/baykus
!git log --oneline -3
!pip -q install uv kaggle
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!uv pip install --system -q -e '/content/baykus/python[train]'
sys.path.insert(0, '/content/baykus/python/src')

In [ ]:
%%time
# BÖLÜM A · ≈ 15-25 dk · M3OT indirme (figshare API, çerez gerekmez) + yalnız train dönüştürme
# Drive'da dönüştürülmüş train varsa indirme atlanır.
import os, subprocess
ZIP, RAW = '/content/M3OT.zip', '/content/m3ot_raw'
MD5 = '7a50499cf07092d1a0e5665048fafbdd'   # M3OT.zip, 10 738 701 490 bayt
if os.path.isdir(f'{OUT}/ir/images/train'):
    print('dönüştürülmüş train zaten açık:', OUT)
elif os.path.exists(CACHE):
    !mkdir -p {OUT} && unzip -q -o {CACHE} -d {OUT}
    print('Drive önbelleğinden açıldı:', CACHE)
else:
    !curl -fL --retry 3 -sS -o {ZIP} https://api.figshare.com/v2/file/download/52023875
    got = subprocess.run(['md5sum', ZIP], capture_output=True, text=True).stdout.split()[0]
    if got != MD5:
        raise RuntimeError(f"M3OT.zip MD5 tutmadı ({got}); bu hücrenin çıktısını Claude'a getir")
    !unzip -q -n {ZIP} -d {RAW} && rm -f {ZIP}
    !baykus-data m3ot --raw {RAW} --out {OUT} --splits train --workers {NPROC}
    !rm -rf {RAW}
    !cd {OUT} && zip -q -r -0 {CACHE} rgb ir labels_ir labels_rgb meta
!cat {OUT}/meta/m3ot_stats.json
!du -sh {OUT}/* {CACHE}

In [ ]:
%%time
# BÖLÜM A · ≈ 20-40 dk (CPU ORT) · öğretmen etiketleri: bugünkü E3c (Release models-v1)
!bash scripts/fetch_models.sh /content/models configs/engine/models.json
!cd python && baykus-data m3ot-teacher --root {OUT} --split train --models /content/models --manifest ../configs/engine/models.json --out {DRIVE}/m3ot/teacher.json

In [ ]:
%%time
# BÖLÜM A · ≈ 2 dk · kör denetim sayfası (100 iz) → Drive'a teacher_audit.zip
!rm -rf /content/teacher_audit
!cd python && baykus-data m3ot-audit --root {OUT} --split train --teacher {DRIVE}/m3ot/teacher.json --out /content/teacher_audit
!rm -f {DRIVE}/m3ot/teacher_audit.zip
!cd /content/teacher_audit && zip -q -r {DRIVE}/m3ot/teacher_audit.zip .
!ls -la {DRIVE}/m3ot/teacher_audit.zip {DRIVE}/m3ot/teacher.json

🚧 **DUR (Bölüm A bitti).** Claude'a hücre 3-5'in son satırlarını getir. Sonra laptopta:
1. Drive'dan `baykus/m3ot/teacher_audit.zip` ve `baykus/m3ot/teacher.json` dosyalarını indir.
2. Zip'i bir klasöre aç, `index.html`'i tarayıcıda aç; her aracın türünü seç (emin değilsen "bilmiyorum").
3. Sayfanın altındaki "cevapları indir" düğmesiyle cevap dosyasını indir ve Claude'a ver.

Claude doğruluğu ölçüp kararı (`teacher` ya da `all_car`) söyler; o kararı hücre 6'ya yaz.

In [ ]:
%%time
# BÖLÜM B · ≈ 2 dk · Claude'un denetim kararıyla M3OT train etiketleri
DECISION = 'teacher'   # kör denetim 0,92 ≥ 0,80 (2026-10-26, results/data/m3ot_teacher_audit.json)
assert DECISION in ('teacher', 'all_car'), "DECISION'ı Claude'un verdiği değerle doldur"
import os
if not os.path.isdir(f'{OUT}/ir/images/train'):
    !mkdir -p {OUT} && unzip -q -o {CACHE} -d {OUT}
FLAG = f'--teacher {DRIVE}/m3ot/teacher.json' if DECISION == 'teacher' else '--all-car'
!cd python && baykus-data m3ot-relabel --root {OUT} --split train {FLAG}

In [ ]:
%%time
# BÖLÜM B · ≈ 30-45 dk · DroneVehicle (Kaggle) + M3OT train birleştirme + 4 kanallı TIFF
import os, shutil
if not os.path.isdir(f'{DV}/rgb/images/train'):
    !kaggle datasets download ouzhantarhan/baykus-dv-rgbt -p {DV} --unzip -q
    !mv {DV}/meta /content/meta
    !baykus-data illum-relabel --meta /content/meta
    !baykus-data slices --out {DV} --meta /content/meta
    !baykus-data yaml --out {DV}
if not any(p.startswith('m3ot_') for p in os.listdir(f'{DV}/ir/images/train')):
    !baykus-data m3ot-merge --m3ot {OUT} --into {DV} --split train
n_m3ot = sum(p.startswith('m3ot_') for p in os.listdir(f'{DV}/ir/images/train'))
print('eğitimdeki M3OT çifti:', n_m3ot)   # 8630 beklenir
assert n_m3ot > 0, 'M3OT birleşmedi: önce hücre 6 (M3OT Drive önbelleğinden açılır)'
!baykus-data rgbt4 --out {DV} --splits train val --workers {NPROC}
os.makedirs('/content/e3c', exist_ok=True)
shutil.copy(f'{DRIVE}/runs/{E3C_RUN}/weights/best.pt', '/content/e3c/yolo26s.pt')   # configs/e3c_m3ot.yaml `model`
!ls {DV}/rgbt4/images/train | wc -l
!du -sh {DV}/rgbt4
!df -h /content

In [ ]:
%%time
# BÖLÜM B · birkaç saat · ince ayar (configs/e3c_m3ot.yaml, 20 epoch, patience 10).
# Her epoch Drive'a last.pt; oturum koparsa hücre 1-2, 6-7, sonra bu hücre: kaldığı yerden sürer.
import os, torch
assert torch.cuda.is_available(), 'GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU'
n_m3ot = sum(p.startswith('m3ot_') for p in os.listdir(f'{DV}/ir/images/train'))
assert n_m3ot > 0, 'M3OT eğitim verisinde yok: önce hücre 6, sonra hücre 7'
print('GPU:', torch.cuda.get_device_name(0), '· eğitimdeki M3OT çifti:', n_m3ot)
from baykus.train.checkpoints import find_runs, status_report
EXP, CONFIG = 'e3c_m3ot', 'configs/e3c_m3ot.yaml'
todo = find_runs(f'{DRIVE}/runs', EXP, finished=False)
done = find_runs(f'{DRIVE}/runs', EXP, finished=True)
if todo:
    RUN_DIR = str(todo[-1])
    print('yarım kalan eğitim, devam:', RUN_DIR)
    !cd /content/baykus/python && baykus-train --config ../{CONFIG} --resume "{RUN_DIR}" --data-root {DV} --results {DRIVE}/results
elif done:
    print('biten eğitim var, yeniden eğitilmedi:', done[-1])
else:
    !cd /content/baykus/python && baykus-train --config ../{CONFIG} --data-root {DV} --project {DRIVE}/runs --results {DRIVE}/results
print(status_report(f'{DRIVE}/runs', EXP))

In [ ]:
%%time
# BÖLÜM B · ≈ 10-15 dk · ONNX FP32 + DroneVehicle val (sevk kuralı i: E3c 0,8411'den ≤ 1,0 puan düşüş,
# sınıf başına ≤ 3,0 puan). 'map50' ve 'per_class' satırlarını Claude'a getir.
import json, yaml
from pathlib import Path
from baykus.train.checkpoints import find_runs
RUN = find_runs(f'{DRIVE}/runs', 'e3c_m3ot', finished=True)[-1].name
EDGE_CFG = '/content/baykus/configs/edge/e11_edge.yaml'
EDGE = {'experiment': 'e11_edge', 'imgsz': 640, 'seed': 42,
        'calibration': {'split': 'train', 'n': 500}, 'variants': ['onnx_fp32'],
        'models': {'e3c_m3ot': {'run_id': RUN, 'config': 'configs/e3c_m3ot.yaml'}},
        'pipelines': {'e3c_m3ot': {'kind': 'single', 'model': 'e3c_m3ot'}},
        'bench': {'warmup': 50, 'runs': 500, 'images': 20, 'conf': 0.25},
        'rule': {'target_p95_ms': 100, 'max_map50_drop': 0.010, 'order': ['e3c_m3ot']}}
Path(EDGE_CFG).write_text(yaml.safe_dump(EDGE, allow_unicode=True), encoding='utf-8')
MODELS_DIR = f'{DRIVE}/edge/models_e11'
# repo_root() yapılandırmanın üstünde 'configs' arar: dosya depo içinde (commit edilmez)
W = f'{DRIVE}/runs/{RUN}/weights/best.pt'
!cd /content/baykus/python && baykus-edge export --config {EDGE_CFG} --model e3c_m3ot --weights "{W}" --out {MODELS_DIR} --variants onnx_fp32 --data-root {DV}
ONNX = f'{MODELS_DIR}/{RUN}_onnx_fp32.onnx'
!cd /content/baykus/python && baykus-edge eval --config {EDGE_CFG} --model e3c_m3ot --variant onnx_fp32 --file "{ONNX}" --data-root {DV} --meta /content/meta --project {DRIVE}/edge/eval_e11 --results {DRIVE}/results --threads {NPROC}
!ls {DRIVE}/results/edge | grep {RUN}

In [ ]:
%%time
# BÖLÜM B · ≈ 15-20 dk · M3OT val (1_08, 2_08) kapsamı: E3c ve E3c-M aynı makinede, aynı komutla
# (sevk kuralı ii: AP50 artmalı). 'birleşik/clean' satırlarını Claude'a getir.
import os, shutil
if not os.path.isdir('/content/models'):   # oturum yenilendiyse (hücre 4 atlandıysa) E3c
    !bash scripts/fetch_models.sh /content/models configs/engine/models.json
if not os.path.isdir('/content/dev_pack/sequences'):
    !mkdir -p /content/dev_pack && unzip -q -o {DRIVE}/m3ot/dev_pack.zip -d /content/dev_pack
os.makedirs('/content/models_m', exist_ok=True)
for f in ('thresholds.json', 'calibration.json', 'gate_model.json'):
    shutil.copy(f'/content/models/{f}', f'/content/models_m/{f}')
shutil.copy(ONNX, '/content/models_m/e3c.onnx')
SEQS = '/content/dev_pack/sequences/1_08 /content/dev_pack/sequences/2_08'
for name, mdir in (('e3c', '/content/models'), ('e3c_m3ot', '/content/models_m')):
    !cd python && baykus-engine coverage --models {mdir} --seq-dir {SEQS} --labels /content/dev_pack/labels --conds clean --out {DRIVE}/results/m3ot_dev_coverage_{name}.json